In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/vehicle-type-classification/vehicle_train_10000.csv
/kaggle/input/competitions/vehicle-type-classification/vehicle_test.csv
/kaggle/input/competitions/vehicle-type-classification/vehicle_sample_submission.csv


In [2]:
import glob, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold

def find(pattern):
    return glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)[0]

train = pd.read_csv(find("vehicle_train*.csv"))
test  = pd.read_csv(find("vehicle_*test*.csv"))
sample = pd.read_csv(find("vehicle_sample_submission.csv"))
print(train.shape, test.shape)

(10000, 22) (2500, 21)


In [3]:
features = [c for c in train.columns if c not in ("vehicle_id", "vehicle_class")]

# duplicates are judged on the features + label, NOT on vehicle_id
print("duplicate rows:", train.drop(columns="vehicle_id").duplicated().sum())
clean = train.drop(columns="vehicle_id").drop_duplicates().reset_index(drop=True)

X = clean[features].values
y = clean["vehicle_class"].values
sd = X.std(axis=0)
print("clean shape:", clean.shape)

duplicate rows: 2500
clean shape: (7500, 21)


In [4]:
def augment(X, y, levels=(0, .15, .2, .25, .3, .2, .2), seed=0):
    """Stack the clean data with copies that have Gaussian noise
    (a fraction of each feature's std) added."""
    rng = np.random.default_rng(seed)
    Xs = [X + rng.normal(size=X.shape) * sd * k for k in levels]
    return np.vstack(Xs), np.tile(y, len(levels))

def make_model():
    # Ridge (L2) multinomial logistic regression
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=0.1, max_iter=3000))

In [5]:
skf = StratifiedKFold(5, shuffle=True, random_state=3)
scores = {0: [], 0.2: []}
for tr, va in skf.split(X, y):
    Xa, ya = augment(X[tr], y[tr])
    model = make_model().fit(Xa, ya)
    for k in scores:
        rng = np.random.default_rng(9)
        Xv = X[va] + rng.normal(size=X[va].shape) * sd * k
        scores[k].append((model.predict(Xv) == y[va]).mean())

print("CV accuracy, clean validation :", np.mean(scores[0]).round(4))
print("CV accuracy, noisy validation :", np.mean(scores[0.2]).round(4))

CV accuracy, clean validation : 0.8936
CV accuracy, noisy validation : 0.8901


In [6]:
Xa, ya = augment(X, y)
final = make_model().fit(Xa, ya)
pred = final.predict(test[features].values)
print(pd.Series(pred).value_counts())

0    843
2    829
1    828
Name: count, dtype: int64


In [7]:
submission = pd.DataFrame({"vehicle_id": test["vehicle_id"],
                           "vehicle_class": pred})
submission.to_csv("submission.csv", index=False)
print(submission.shape, (submission.vehicle_id == sample.vehicle_id).all())
submission.head()

(2500, 2) True


,vehicle_id,vehicle_class
0,10001,0
1,10002,0
2,10003,1
3,10004,2
4,10005,1
